# RQ2: single- vs multi-source fine-tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syndr0/compute-efficient-genqa/blob/main/notebooks/02_rq2_generalization.ipynb)

In [ ]:
import os
import sys

import torch

USE_DRIVE = True
SMOKE_TEST = False
GPU_PROFILE = "auto"  # auto | t4 | a100

REPO_DIR = "/content/compute-efficient-genqa"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Syndr0/compute-efficient-genqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cegqa_outputs"
else:
    OUT = "/content/cegqa_outputs"

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
if GPU_PROFILE == "auto":
    GPU_PROFILE = "a100" if GPU_MEM_GB >= 39 else "t4"
CONFIG_FILES = ["configs/default.yaml"]
if GPU_PROFILE == "a100":
    CONFIG_FILES.append("configs/a100.yaml")
    OUT += "_a100"
if SMOKE_TEST:
    CONFIG_FILES.append("configs/smoke.yaml")
    OUT += "_smoke"
os.environ["CEGQA_OUTPUT_DIR"] = OUT
CONFIG = " ".join(f"--config {f}" for f in CONFIG_FILES)
print(f"GPU: {GPU} ({GPU_MEM_GB:.0f} GB) | profile: {GPU_PROFILE} | outputs: {OUT}")

In [ ]:
!python -m cegqa.cli prepare $CONFIG

In [ ]:
!python -m cegqa.cli rq2 $CONFIG

In [ ]:
!python -m cegqa.cli report $CONFIG

In [ ]:
import json

import pandas as pd
from IPython.display import Image, Markdown, display

from cegqa.data import eval_sources
from cegqa.utils import load_config, read_jsonl

cfg = load_config(CONFIG_FILES)

In [ ]:
st = json.load(open(f"{cfg['output_dir']}/analysis/rq2_stats.json"))
base, treat = list(st["conditions"])
rows = []
for src, e in st["by_source"].items():
    rows.append({"eval": src, "ood": e["ood"],
                 f"f1_{base}": f"{e[base]['f1']['mean']:.1f} ± {e[base]['f1']['std']:.1f}",
                 f"f1_{treat}": f"{e[treat]['f1']['mean']:.1f} ± {e[treat]['f1']['std']:.1f}",
                 "delta_f1": round(e["paired_f1"]["diff"], 2),
                 "ci95": f"[{e['paired_f1']['lo']:+.1f}, {e['paired_f1']['hi']:+.1f}]",
                 "p": round(e["paired_f1"]["p_two_sided"], 4)})
pd.DataFrame(rows)

In [ ]:
pd.DataFrame([{"eval": src, "condition": cond, **{k: round(100 * v, 1) for k, v in e[cond]["errors"]["categories"].items()}}
              for src, e in st["by_source"].items() for cond in (base, treat)])

In [ ]:
for f in ['rq2_f1_bars.png']:
    display(Image(f"{cfg['output_dir']}/figures/{f}"))